In [13]:
import pandas as pd
# test = pd.read_parquet('/Users/patrickmaloney/Documents/sports-trading/mlb-betting/data/raw/lineups/2026-02-24/lineups_013149.parquet')
pd.set_option('display.max_columns', None)

In [ ]:
pd.read_parquet('/Users/patrickmaloney/Documents/sports-trading/mlb-betting/data/raw/schedules/games_2000.parquet')

In [ ]:
# =============================================
# NOTEBOOK SETUP — Run this cell first
# =============================================
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent          # goes up from notebooks/
sys.path.insert(0, str(PROJECT_ROOT))

import duckdb
from config.settings import DB_PATH

print("✅ Project root:", PROJECT_ROOT)
print("✅ Connecting to DuckDB (read-only):", DB_PATH)

# read_only=True = no more lock errors for analysis
con = duckdb.connect(str(DB_PATH), read_only=True)

print("✅ Connected successfully in read-only mode!")
print("Total games in silver_lineups:", 
      con.sql("SELECT COUNT(*) as total FROM silver_lineups").df().iloc[0]['total'])

Day By Day Stat Download Testing

In [ ]:
# ────────────────────────────────────────────────
# Cell 1: Basic imports + one-day batting totals
# ────────────────────────────────────────────────

import pybaseball as pb
import pandas as pd

# Example: get all MLB batting stats that occurred on July 4, 2024
start_date = "2025-02-20"
# end_date   = "2024-07-04"   # same date = single day
end_date = start_date

daily_batting = pb.batting_stats_range(start_date, end_date)

print(f"Shape: {daily_batting.shape}")
display(daily_batting.head(8))          # first few rows (player-level)
print("\nColumns:", daily_batting.columns.tolist())

# Quick team totals for that day
team_totals = daily_batting.groupby('Tm').agg({
    'G': 'sum', 'PA': 'sum', 'AB': 'sum', 'R': 'sum', 'H': 'sum',
    'HR': 'sum', 'RBI': 'sum', 'BB': 'sum', 'SO': 'sum'
}).sort_values('R', ascending=False)

print("\nTeam Batting Totals on", start_date)
display(team_totals)

In [ ]:
# ────────────────────────────────────────────────
# Cell 2: Same thing but for pitching stats on a date
# ────────────────────────────────────────────────

daily_pitching = pb.pitching_stats_range(start_date, end_date)

team_pitch_totals = daily_pitching.groupby('Tm').agg({
    'G': 'sum', 'IP': 'sum', 'H': 'sum', 'R': 'sum', 'ER': 'sum',
    'HR': 'sum', 'BB': 'sum', 'SO': 'sum', 'ERA': 'mean'   # or weighted later
}).sort_values('R', ascending=True)   # lower runs allowed = better

print("\nTeam Pitching Totals Allowed on", start_date)
display(team_pitch_totals)

In [ ]:
# ────────────────────────────────────────────────
# Cell 3: One-liner function you can reuse
# ────────────────────────────────────────────────

def get_daily_team_totals(date_str: str, stat_type='batting'):
    """
    date_str: 'YYYY-MM-DD'
    stat_type: 'batting' or 'pitching'
    """
    start = end = date_str
    
    if stat_type == 'batting':
        df = pb.batting_stats_range(start, end)
        key_stats = ['G','PA','AB','R','H','2B','3B','HR','RBI','BB','SO']
    elif stat_type == 'pitching':
        df = pb.pitching_stats_range(start, end)
        key_stats = ['G','IP','H','R','ER','HR','BB','SO']
    else:
        raise ValueError("stat_type must be 'batting' or 'pitching'")
    
    if df.empty:
        print(f"No data found for {date_str}")
        return pd.DataFrame()
    
    team_df = df.groupby('Tm', as_index=False)[key_stats].sum()
    team_df = team_df.sort_values('R', ascending=(stat_type=='pitching'))
    
    print(f"{stat_type.title()} team totals for {date_str} — {len(team_df)} teams")
    return team_df

# Usage examples
display(get_daily_team_totals("2025-08-15", "batting"))
# display(get_daily_team_totals("2023-10-01", "pitching"))

In [82]:
pd.read_parquet('/Users/patrickmaloney/Documents/mlb-betting/data/raw/schedules/games_2026.parquet')
batpath = '/Users/patrickmaloney/Documents/mlb-betting/data/raw/player_logs/game_by_game/batting_game_logs_2025.parquet'
batting = pd.read_parquet(batpath)
lineup = pd.read_parquet('/Users/patrickmaloney/Documents/mlb-betting/data/bronze/lineups/2026-03-08/lineups_201831.parquet')
lahman = pd.read_parquet('/Users/patrickmaloney/Documents/mlb-betting/data/reference/lahman_files/people.parquet')

In [61]:
lahman.head(4).to_clipboard()
# to make it maybe a little faster, not sure if its doing this already but when loading the parquet cant you filter for just the correct player ids?

In [ ]:
pd.read_parquet('/Users/patrickmaloney/Documents/mlb-betting/data/simulations/simulation_results_20260308_1647.parquet')
# display(pd.read_parquet('/Users/patrickmaloney/Documents/mlb-betting/data/simulations/simulation_results_20260308_1721.parquet'))
display(pd.read_parquet('/Users/patrickmaloney/Documents/mlb-betting/data/simulations/simulation_results_20260308_1736.parquet'))

In [ ]:
batting#[batting.]

In [ ]:
pd.read_parquet('/Users/patrickmaloney/Documents/mlb-betting/data/reference/lahman_files/historical_teams_data.parquet')[['yearID', 'lgID', 'teamID', 'divID', 'teamIDBR']].drop_duplicates()

Game Sim Calc

In [ ]:
# =============================================
# CELL 1: Imports + Load Data (run once)
# =============================================
import pandas as pd
import numpy as np
import glob
import ast
from datetime import datetime
from rapidfuzz import process

print("🚀 Loading historical batting logs...")
logs_dir = "/Users/patrickmaloney/Documents/mlb-betting/data/raw/player_logs/game_by_game"
bat_files = sorted(glob.glob(f"{logs_dir}/batting_game_logs_*.parquet"))
batting = pd.concat([pd.read_parquet(f) for f in bat_files], ignore_index=True)
print(f"✅ Loaded {len(batting):,} batting rows")

# Name → mlbID fallback map
name_to_mlb = {}
for _, row in batting.iterrows():
    name = str(row.get('Name', '')).strip().lower()
    mlbid = row.get('mlbID')
    if name and pd.notna(mlbid):
        name_to_mlb[name] = mlbid

# wOBA calculation (if missing)
if 'wOBA' not in batting.columns:
    print("   Calculating wOBA from linear weights...")
    lw = pd.read_csv("/Users/patrickmaloney/Documents/mlb-betting/data/reference/linear_weights.csv").set_index('Season')
    def calc_woba(row):
        year = int(row.get('game_year', 2025))
        if year not in lw.index:
            year = lw.index.max()
        w = lw.loc[year]
        pa = row.get('PA', 1) or 1
        num = (w.get('wBB', 0.69) * row.get('BB', 0) +
               w.get('wHBP', 0.72) * row.get('HBP', 0) +
               w.get('w1B', 0.88) * (row.get('H', 0) - row.get('2B', 0) - row.get('3B', 0) - row.get('HR', 0)) +
               w.get('w2B', 1.25) * row.get('2B', 0) +
               w.get('w3B', 1.57) * row.get('3B', 0) +
               w.get('wHR', 2.0) * row.get('HR', 0))
        return num / pa if pa > 0 else 0.320
    batting['wOBA'] = batting.apply(calc_woba, axis=1)

player_woba = batting.groupby('mlbID')['wOBA'].mean().to_dict()
print(f"✅ Built wOBA dictionary for {len(player_woba):,} players")

people = pd.read_parquet("/Users/patrickmaloney/Documents/mlb-betting/data/reference/lahman_files/people.parquet")
print("✅ Reference data loaded\n")

In [ ]:
# =============================================
# CELL 2: Load latest lineups
# =============================================
lineups_dir = "/Users/patrickmaloney/Documents/mlb-betting/data/bronze/lineups"
files = sorted(glob.glob(f"{lineups_dir}/*/*.parquet"), reverse=True)
if not files:
    raise FileNotFoundError("No lineup files — run lineups.py first!")

lineups_df = pd.read_parquet(files[0])
print(f"📋 LOADED {len(lineups_df)} games from:\n{files[0]}\n")
lineups_df.head(2)  # shows you the raw data

In [90]:
# =============================================
# CELL 3: Helper functions
# =============================================
def get_mlbid_list(bbref_list, name_list, people, name_to_mlb):
    mlbids = []
    # 1. BBRef → MLB ID (most accurate)
    for bbref in bbref_list:
        if pd.notna(bbref):
            match = people[people['playerID'] == bbref]
            if not match.empty:
                mlbid = match.iloc[0].get('mlbID') or match.iloc[0].get('key_mlbam')
                if pd.notna(mlbid):
                    mlbids.append(mlbid)
                    continue
    # 2. Fuzzy name match as fallback
    for name in name_list:
        if not name: continue
        clean = str(name).strip().lower()
        if clean in name_to_mlb:
            mlbids.append(name_to_mlb[clean])
            continue
        match = process.extractOne(clean, name_to_mlb.keys(), score_cutoff=75)
        if match:
            mlbids.append(name_to_mlb[match[0]])
    return [m for m in mlbids if m is not None]

def simulate_game_detailed(row, player_woba, people, name_to_mlb):
    print(f"\n{'='*60}")
    print(f"🎮 SIMULATING: {row['away_team']} @ {row['home_team']} — {row.get('game_date')}")
    print(f"{'='*60}")

    # === 1. Parse lineups ===
    try:
        away_bbref = ast.literal_eval(row.get('away_lineup_bbref_ids', '[]'))
        home_bbref = ast.literal_eval(row.get('home_lineup_bbref_ids', '[]'))
        away_names = ast.literal_eval(row.get('away_lineup', '[]'))
        home_names = ast.literal_eval(row.get('home_lineup', '[]'))
        away_pos   = ast.literal_eval(row.get('away_lineup_pos', '[]'))
        home_pos   = ast.literal_eval(row.get('home_lineup_pos', '[]'))
    except:
        away_bbref = home_bbref = []
        away_names = row.get('away_lineup', [])
        home_names = row.get('home_lineup', [])
        away_pos = home_pos = ['?']*9

    print(f"\n📋 AWAY LINEUP ({row['away_team']})")
    for i, (name, pos) in enumerate(zip(away_names, away_pos), 1):
        print(f"   {i:2}. {pos:3} {name}")

    print(f"\n📋 HOME LINEUP ({row['home_team']})")
    for i, (name, pos) in enumerate(zip(home_names, home_pos), 1):
        print(f"   {i:2}. {pos:3} {name}")

    # === 2. Match to MLB IDs + show wOBA ===
    away_mlbid = get_mlbid_list(away_bbref, away_names, people, name_to_mlb)
    home_mlbid = get_mlbid_list(home_bbref, home_names, people, name_to_mlb)

    print(f"\n🔍 MATCHING RESULTS")
    print(f"   Away matched: {len(away_mlbid)}/{len(away_names)} players")
    print(f"   Home matched: {len(home_mlbid)}/{len(home_names)} players")

    def team_woba(ids):
        wobas = [player_woba.get(mid, 0.305) for mid in ids]
        return np.mean(wobas) if wobas else 0.320

    away_woba = team_woba(away_mlbid)
    home_woba = team_woba(home_mlbid)

    print(f"\n📊 TEAM wOBA")
    print(f"   {row['away_team']:3} → {away_woba:.3f}")
    print(f"   {row['home_team']:3} → {home_woba:.3f}")

    # === 3. Run Monte Carlo ===
    away_rs_mean = away_woba * 5.15 * 0.98   # ← explained above
    home_rs_mean = home_woba * 5.15 * 1.04

    n_sims = 10000
    np.random.seed(42)
    away_rs = np.random.normal(away_rs_mean, 2.6, n_sims).clip(0)
    home_rs = np.random.normal(home_rs_mean, 2.6, n_sims).clip(0)

    avg_away = away_rs.mean()
    avg_home = home_rs.mean()
    win_prob = (away_rs > home_rs).mean()
    total = avg_away + avg_home

    print(f"\n🎲 SIMULATION RESULTS (10,000 sims)")
    print(f"   {row['away_team']} projected runs: {avg_away:.2f}")
    print(f"   {row['home_team']} projected runs: {avg_home:.2f}")
    print(f"   Total runs: {total:.2f}")
    print(f"   {row['away_team']} win probability: {win_prob:.1%}")

    return {
        'game_date': row.get('game_date'),
        'away_team': row['away_team'],
        'home_team': row['home_team'],
        'away_rs_proj': round(avg_away, 2),
        'home_rs_proj': round(avg_home, 2),
        'total_proj': round(total, 2),
        'away_win_prob': round(win_prob, 4)
    }

In [ ]:
# =============================================
# CELL 4: Run simulation on ONE game (or loop)
# =============================================
# Change the index below to see any game you want (0 = first game)
game_idx = 0

row = lineups_df.iloc[game_idx]
result = simulate_game_detailed(row, player_woba, people, name_to_mlb)

print("\n" + "="*60)
print("✅ SINGLE GAME DONE!")
print(result)

In [ ]:
home_lineup = lineups_df.iloc[0].home_lineup_bbref_ids
home_lineup

In [ ]:
# =============================================
# NEW CELL: Filter batting logs for ONE lineup using BBRef IDs
# (Run this after your previous cells — batting, people, lineups_df are already loaded)
# =============================================

import numpy as np
import ast

# Get the BBRef list (it's already a numpy array from your lineups)
home_bbref = lineups_df.iloc[0]['home_lineup_bbref_ids']

# Make sure it's a clean Python list
if isinstance(home_bbref, np.ndarray):
    home_bbref = home_bbref.tolist()
elif isinstance(home_bbref, str):
    home_bbref = ast.literal_eval(home_bbref)

print(f"🔑 Raw BBRef IDs from lineup: {home_bbref}\n")

# === MAP BBRef IDs → mlbID + real names (EXACT match via people table) ===
# We use the SAME logic as get_mlbid_list, but we don't need fuzzy matching here
# (fuzzy is only a fallback when we have *names only*)
mlb_ids = []
player_map = {}

for bbref in home_bbref:
    if pd.isna(bbref) or not bbref:
        continue
    match = people[people['playerID'] == bbref]
    if not match.empty:
        row = match.iloc[0]
        mlbid = row.get('mlbID') or row.get('key_mlbam')
        if pd.notna(mlbid):
            full_name = f"{row.get('nameFirst', '')} {row.get('nameLast', '')}".strip()
            mlb_ids.append(mlbid)
            player_map[bbref] = {'mlbID': mlbid, 'name': full_name}

print("✅ Exact BBRef → mlbID Mapping:")
for bbref, info in player_map.items():
    print(f"   {bbref:10} → {info['mlbID']}  ({info['name']})")

# === FILTER the huge batting dataframe to ONLY these players ===
home_batting = batting[batting['mlbID'].isin(mlb_ids)].copy()

print(f"\n📊 FILTERED: {len(home_batting):,} rows (game logs) for these {len(player_map)} players")

# === SHOW NICE STATS SUMMARY ===
summary = (
    home_batting
    .groupby('Name')                          # Name column exists in your batting logs
    .agg(
        mlbID=('mlbID', 'first'),
        games=('mlbID', 'count'),             # one row per game
        total_PA=('PA', 'sum'),
        avg_wOBA=('wOBA', 'mean'),
        last_game=('game_date', 'max') if 'game_date' in home_batting.columns else ('mlbID', 'count')  # fallback
    )
    .round(3)
    .sort_values('avg_wOBA', ascending=False)
)

display(summary)

In [ ]:
# =============================================
# UPDATED: BBRef → mlbID + LEAGUE AVERAGE FALLBACK
# =============================================

import numpy as np
import ast
from rapidfuzz import process

# 1. Get the BBRef list (clean it)
home_bbref = lineups_df.iloc[0]['home_lineup_bbref_ids']
if isinstance(home_bbref, np.ndarray):
    home_bbref = home_bbref.tolist()
elif isinstance(home_bbref, str):
    home_bbref = ast.literal_eval(home_bbref)

print(f"🔑 BBRef IDs from lineup: {home_bbref}\n")

# 2. Calculate TRUE league average wOBA from your entire dataset
league_avg_woba = round(batting['wOBA'].mean(), 3)
print(f"🏆 League average wOBA (fallback for missing players): {league_avg_woba}\n")

# 3. Build BBRef → mlbID mapping (same as before)
bbref_to_mlbid = {}
bbref_to_name = {}   # for nice display

for bbref in home_bbref:
    if pd.isna(bbref) or not bbref:
        continue
    match = people[people['playerID'] == str(bbref)]
    if match.empty:
        print(f"❌ No entry in people for {bbref}")
        continue
    row = match.iloc[0]
    first = str(row.get('nameFirst', '')).strip()
    last = str(row.get('nameLast', '')).strip()
    full_name = f"{first} {last}".strip().lower()
    
    if not full_name:
        continue
        
    result = process.extractOne(full_name, name_to_mlb.keys(), score_cutoff=85)
    if result:
        matched_name, score = result[0], result[1]
        mlbid = name_to_mlb[matched_name]
        bbref_to_mlbid[bbref] = mlbid
        bbref_to_name[bbref] = matched_name.title()
        print(f"✅ {bbref:10} → {matched_name.title():25} → mlbID {mlbid} (score: {score})")
    else:
        print(f"❌ Could not fuzzy-match {bbref} ({full_name})")

print(f"\n✅ Successfully mapped {len(bbref_to_mlbid)}/{len([b for b in home_bbref if pd.notna(b)])} players\n")

# 4. Filter real data + add fallback rows for missing players
mlbid_list = list(bbref_to_mlbid.values())
real_data = batting[batting['mlbID'].isin(mlbid_list)].copy()

# Build full summary that includes EVERYONE (with league avg fallback)
summary_rows = []
for bbref in home_bbref:
    if pd.isna(bbref):
        continue
    name = bbref_to_name.get(bbref, "Unknown Player")
    if bbref in bbref_to_mlbid:
        mlbid = bbref_to_mlbid[bbref]
        player_logs = real_data[real_data['mlbID'] == mlbid]
        if not player_logs.empty:
            games = len(player_logs)
            total_pa = player_logs['PA'].sum()
            avg_woba = round(player_logs['wOBA'].mean(), 3)
            last_game = player_logs['game_date'].max() if 'game_date' in player_logs.columns else "—"
        else:
            games = 0
            total_pa = 0
            avg_woba = league_avg_woba
            last_game = "NO DATA — using league avg"
    else:
        # Never even found in people table
        games = 0
        total_pa = 0
        avg_woba = league_avg_woba
        last_game = "NO MATCH — using league avg"
    
    summary_rows.append({
        'Name': name,
        'BBRef_ID': bbref,
        'mlbID': bbref_to_mlbid.get(bbref, "—"),
        'games': games,
        'total_PA': total_pa,
        'avg_wOBA': avg_woba,
        'last_game': last_game
    })

summary = pd.DataFrame(summary_rows).sort_values('avg_wOBA', ascending=False)
display(summary)

In [ ]:
# =============================================
# CORRECTED: Realistic lineup run projection
# =============================================

# Realistic PA per batting-order spot
pa_per_spot = [4.85, 4.70, 4.55, 4.45, 4.35, 4.25, 4.15, 4.05, 3.95]

# CORRECT scaling factor (this was the bug before)
scaling = 5.15 / (0.320 * 38.5)   # ≈ 0.418

summary = summary.copy()  # don't overwrite old one
summary['batting_order'] = range(1, len(summary)+1)
summary['expected_PA'] = pa_per_spot[:len(summary)]

summary['runs_contributed'] = round(
    summary['avg_wOBA'] * summary['expected_PA'] * scaling, 3
)

total_projected_runs = round(summary['runs_contributed'].sum(), 2)

print(f"🏟️ PROJECTED RUNS FOR THIS LINEUP: **{total_projected_runs}** (realistic!)")
display(summary[['Name', 'batting_order', 'expected_PA', 'avg_wOBA', 'runs_contributed']])
print(f"\nTotal Projected Runs: {total_projected_runs}")

In [ ]:
import pandas as pd
pd.read_parquet('/Users/patrickmaloney/Documents/mlb-betting/data/raw/odds/2026-03-13/games/odds_122005.parquet')

: 

In [ ]:
batting